In [ ]:

import pandas as pd

# Read the first CSV file into a DataFrame
df_vrs_repairs = pd.read_csv("vrs_correcoes.csv")

df_vrs_repairs

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['Constraint Deleted'] == True)] )

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['Constraint Deprecated'] == True)] )

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['Included as Exception'] == True)] )

In [ ]:
df_vrs_repairs['A-box wdt statement Deleted'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def statementDeleted(row):
    
    if row['object'].startswith("_:"):
        return None
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{wdt_pid}> <{row['object']}>  }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
print(statementDeleted(df_vrs_repairs.iloc[0]))

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['A-box wdt statement Deleted']):
        result = statementDeleted(row)
        df_vrs_repairs.at[index, 'A-box wdt statement Deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 10000 == 0:
        df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_vrs_repairs

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['A-box wdt statement Deleted'] == True)] )

In [ ]:
df_vrs_repairs['A-box wdt required statement added'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def onlyReqPropStmtAdded(row):
    
    if row['object'].startswith("_:"):
        return None
    
    if row['2019_no_req_val'] is False:
        return None
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['object']}> <{row['wdt_required_property']}> []}}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
print(onlyReqPropStmtAdded(df_vrs_repairs.iloc[0]))

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getRequiredValues(row, endpoint = "ENTER_qEndpoint_WD_2019"):
    
    # URL of the endpoint
    #endpoint = "ENTER_qEndpoint_WD_2023"
    if row['object'].startswith("_:"):
        return None
    
    if row['2019_no_req_val'] is True:
        return None
    
    wd_required_pid = row['wdt_required_property'].replace("http://www.wikidata.org/prop/direct/", "http://www.wikidata.org/entity/")
    
    # SPARQL query
    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>

        SELECT 
          ?value
        WHERE
        {{
          <{row['property']}> p:P2302 ?statement.
          ?statement ps:P2302 wd:Q21510864.
          ?statement pq:P2306 <{wd_required_pid}>.
          ?statement pq:P2305 ?value. 
        }}
    """
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find all 'uri' elements and extract their text
        uris = [uri_element.text for uri_element in root.findall('.//ns:uri', namespace)]

        if len(uris) == 0:
            return []
        
        return uris
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
    
def reqPropValStmtAdded(row, required_value):
    
    if row['object'].startswith("_:"):
        return None
    
    if row['2019_no_req_val'] is True:
        return None
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['object']}> <{row['wdt_required_property']}> <{required_value}>}}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
def testReqStmtWithReqValueAdded(row):
    reqValues = getRequiredValues(row)
    if reqValues is None:
        return None
    if len(reqValues) == 0:
        return None
    for reqVal in reqValues:
        result = reqPropValStmtAdded(row, reqVal)
        if result is True:
            return True
    return False

testReqStmtWithReqValueAdded(df_vrs_repairs.iloc[482273])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['A-box wdt required statement added']):
        if row['2019_no_req_val'] is True:
            result = onlyReqPropStmtAdded(row)
            df_vrs_repairs.at[index, 'A-box wdt required statement added'] = result
        else:
            result = testReqStmtWithReqValueAdded(row)
            df_vrs_repairs.at[index, 'A-box wdt required statement added'] = result

    # Save a checkpoint every 10,000 rows
    if index % 300000 == 0:
        df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_vrs_repairs.to_csv("checkpoint_vrs_ok.csv", index=False)

In [ ]:
df_vrs_repairs[(df_vrs_repairs['2019_no_req_val'] == False)]

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['A-box wdt required statement added'] == True)] )

In [ ]:
df_vrs_repairs[
    (df_vrs_repairs['Constraint Deleted'] == False)
    & (df_vrs_repairs['Constraint Deprecated'] == False)
    & (df_vrs_repairs['Included as Exception'] == False)
    & (df_vrs_repairs['A-box wdt statement Deleted'] == False)
    & ((df_vrs_repairs['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs['A-box wdt required statement added'])))
]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def hasDeprecatedReason(row):

    if bool(row['2019_no_req_val']):
        string_req_val = "FILTER NOT EXISTS {{?statement pq:P2305 []}}"
    else:
        string_req_val = "?statement pq:P2305 []."
        

    # SPARQL query
    query = f"""PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX wikibase: <http://wikiba.se/ontology#>
            PREFIX p: <http://www.wikidata.org/prop/>
            PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
            PREFIX ps: <http://www.wikidata.org/prop/statement/>
            PREFIX wd: <http://www.wikidata.org/entity/>
            PREFIX wdt: <http://www.wikidata.org/prop/direct/>

            ASK
            {{
              ?statement ps:P2302 wd:Q21510864. ## value-requires-statement constraint
              <{row['property']}> p:P2302 ?statement.
              ?statement pq:P2306/wikibase:directClaim <{row['wdt_required_property']}>.

              {string_req_val}
              # exception and deprecation
              ?statement pq:P2241 ?rank
            }}
            """
    #print(query)
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
hasDeprecatedReason(df_vrs_repairs.iloc[8096])

In [ ]:
type(df_vrs_repairs.iloc[8096]['Constraint Deprecated'])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    # Check for unprocessed rows
    if bool(row['Constraint Deprecated']) is False:
        result = hasDeprecatedReason(row)
        df_vrs_repairs.at[index, 'Constraint Deprecated'] = result
       

    # Save a checkpoint every 10,000 rows
    if index % 100000 == 0:
        df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['Constraint Deprecated'] == True)] )

In [ ]:
df_vrs_repairs[
    (df_vrs_repairs['Constraint Deleted'] == False)
    & (df_vrs_repairs['Constraint Deprecated'] == False)
    & (df_vrs_repairs['Included as Exception'] == False)
    & (df_vrs_repairs['A-box wdt statement Deleted'] == False)
    & ((df_vrs_repairs['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs['A-box wdt required statement added'])))
]

In [ ]:
df_vrs_repairs

In [ ]:
import requests
import xml.etree.ElementTree as ET

def statementWithBlankObjDeleted(row):
    
    if not row['object'].startswith("_:"):
        return None
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{wdt_pid}> []}}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
print(statementWithBlankObjDeleted(df_vrs_repairs.iloc[1392179]))

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['A-box wdt statement Deleted']) or row['object'].startswith("_:"):
        result = statementWithBlankObjDeleted(row)
        df_vrs_repairs.at[index, 'A-box wdt statement Deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 300000 == 0:
        df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['A-box wdt statement Deleted'] == True)] )

In [ ]:
df_vrs_repairs[
    (df_vrs_repairs['Constraint Deleted'] == False)
    & (df_vrs_repairs['Constraint Deprecated'] == False)
    & (df_vrs_repairs['Included as Exception'] == False)
    & (df_vrs_repairs['A-box wdt statement Deleted'] == False)
    & ((df_vrs_repairs['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs['A-box wdt required statement added'])))
]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def hasStmtWithBlankNode(row, endpoint):
    
    if not row['object'].startswith("_:"):
        return None
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    #endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{wdt_pid}> ?o.
            FILTER(!isIRI(?o) || !STRSTARTS(STR(?o), "http://"))
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
def isStmtWithBlankNodeRemoved(row):
    if hasStmtWithBlankNode(row, "ENTER_qEndpoint_WD_2019") and not hasStmtWithBlankNode(row, "ENTER_qEndpoint_WD_2023"):
        return True
    return False

# Example usage
#print(hasStmtWithBlankNode(df_vrs_repairs.iloc[179], "ENTER_qEndpoint_WD_2019"))
#print(hasStmtWithBlankNode(df_vrs_repairs.iloc[179], "ENTER_qEndpoint_WD_2023"))
isStmtWithBlankNodeRemoved(df_vrs_repairs.iloc[179])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['A-box wdt statement Deleted']) or row['object'].startswith("_:"):
        result = isStmtWithBlankNodeRemoved(row)
        df_vrs_repairs.at[index, 'A-box wdt statement Deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 300000 == 0:
        df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['A-box wdt statement Deleted'] == True)] )

In [ ]:
df_vrs_repairs[
    (df_vrs_repairs['Constraint Deleted'] == False)
    & (df_vrs_repairs['Constraint Deprecated'] == False)
    & (df_vrs_repairs['Included as Exception'] == False)
    & (df_vrs_repairs['A-box wdt statement Deleted'] == False)
    & ((df_vrs_repairs['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs['A-box wdt required statement added'])))
]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isBlankStillViolation(row):
    
    if not row['object'].startswith("_:"):
        return None
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{wdt_pid}> ?o.
            ?o <{row['wdt_required_property']}> []
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
# Example usage
isBlankStillViolation(df_vrs_repairs.iloc[206])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

blank_violations = []
# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    # Check for unprocessed rows
    if row['object'].startswith("_:"):
        if (row['Constraint Deleted'] == False) & (row['Constraint Deprecated'] == False) & (row['Included as Exception'] == False) & (row['A-box wdt statement Deleted'] == False)    & ((row['A-box wdt required statement added'] == False)        | (pd.isna(row['A-box wdt required statement added']))):
            if isBlankStillViolation(row):
                blank_violations.append(index)

In [ ]:
blank_violations[:20]

In [ ]:
len(blank_violations)

In [ ]:
len(df_vrs_repairs)

In [ ]:
# Assuming df_vrs_repairs is your DataFrame and blank_violations is your list of indexes
df_vrs_repairs_cleaned = df_vrs_repairs.drop(index=blank_violations)

# Optionally, you can reset the index if needed
df_vrs_repairs_cleaned.reset_index(drop=True, inplace=True)

In [ ]:
len(df_vrs_repairs_cleaned)

In [ ]:
df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['Constraint Deleted'] == False)
    & (df_vrs_repairs_cleaned['Constraint Deprecated'] == False)
    & (df_vrs_repairs_cleaned['Included as Exception'] == False)
    & (df_vrs_repairs_cleaned['A-box wdt statement Deleted'] == False)
    & ((df_vrs_repairs_cleaned['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['A-box wdt required statement added'])))
]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isBlankStillViolation2(row):
    
    if not row['object'].startswith("_:"):
        return None
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{wdt_pid}> ?o.
            FILTER NOT EXISTS {{?o <{row['wdt_required_property']}> []}}
            FILTER(!isIRI(?o))
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
# Example usage
isBlankStillViolation2(df_vrs_repairs_cleaned.iloc[286])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

blank_violations_2 = []
# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs_cleaned.iterrows(), total=len(df_vrs_repairs_cleaned)):
    
    # Check for unprocessed rows
    if row['object'].startswith("_:"):
        if (row['Constraint Deleted'] == False) & (row['Constraint Deprecated'] == False) & (row['Included as Exception'] == False) & (row['A-box wdt statement Deleted'] == False)    & ((row['A-box wdt required statement added'] == False)        | (pd.isna(row['A-box wdt required statement added']))):
            if isBlankStillViolation2(row):
                blank_violations_2.append(index)

In [ ]:
blank_violations_2[:10]

In [ ]:
df_vrs_repairs_cleaned.iloc[8719]

In [ ]:
# Assuming df_vrs_repairs is your DataFrame and blank_violations is your list of indexes
df_vrs_repairs_cleaned = df_vrs_repairs_cleaned.drop(index=blank_violations_2)

# Optionally, you can reset the index if needed
df_vrs_repairs_cleaned.reset_index(drop=True, inplace=True)

In [ ]:
df_vrs_repairs_cleaned

In [ ]:
df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['Constraint Deleted'] == False)
    & (df_vrs_repairs_cleaned['Constraint Deprecated'] == False)
    & (df_vrs_repairs_cleaned['Included as Exception'] == False)
    & (df_vrs_repairs_cleaned['A-box wdt statement Deleted'] == False)
    & ((df_vrs_repairs_cleaned['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['A-box wdt required statement added'])))
]

In [ ]:
base_path = "violations/vrs_violations/"
pd_no_val_2023 = pd.read_csv(base_path + "vrs_violations_onlyReqProp_2023.txt")
pd_with_val_2023 = pd.read_csv(base_path + "vrs_violations_reqPropVal_2023.txt")

In [ ]:
to_check_list = df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['Constraint Deleted'] == False)
    & (df_vrs_repairs_cleaned['Constraint Deprecated'] == False)
    & (df_vrs_repairs_cleaned['Included as Exception'] == False)
    & (df_vrs_repairs_cleaned['A-box wdt statement Deleted'] == False)
    & ((df_vrs_repairs_cleaned['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['A-box wdt required statement added'])))
].index.tolist()

In [ ]:
# Initialize an empty list to store the indexes
indexes_list = []

# Extract the relevant columns (subject, property, object) from pd_no_val_2023
pd_no_val_subset = pd_no_val_2023[['subject', 'property', 'object']]

# Loop through df_vrs_repairs_cleaned and check if (subject, property, object) exists in pd_no_val_subset
for index, row in df_vrs_repairs_cleaned.iterrows():
    if bool(row['2019_no_req_val']) and index in to_check_list:
        if ((row['subject'], row['property'], row['object']) in pd_no_val_subset.itertuples(index=False, name=None)):
            # Add the index to the list
            indexes_list.append(index)

In [ ]:
indexes_list

In [ ]:
# Assuming df_vrs_repairs is your DataFrame and blank_violations is your list of indexes
df_vrs_repairs_cleaned = df_vrs_repairs_cleaned.drop(index=indexes_list)

# Optionally, you can reset the index if needed
df_vrs_repairs_cleaned.reset_index(drop=True, inplace=True)

In [ ]:
df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['Constraint Deleted'] == False)
    & (df_vrs_repairs_cleaned['Constraint Deprecated'] == False)
    & (df_vrs_repairs_cleaned['Included as Exception'] == False)
    & (df_vrs_repairs_cleaned['A-box wdt statement Deleted'] == False)
    & ((df_vrs_repairs_cleaned['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['A-box wdt required statement added'])))
]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getConstraintID(row, endpoint = "ENTER_qEndpoint_WD_2019"):
        
    #wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    #endpoint = "ENTER_qEndpoint_WD_2019"

    if bool(row['2019_no_req_val']):
        req_val = "FILTER NOT EXISTS {?statement pq:P2305 []}"
    else:
        req_val = "?statement pq:P2305 []."
    
    # SPARQL query
    query = f"""
            PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
            PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX wikibase: <http://wikiba.se/ontology#>
            PREFIX p: <http://www.wikidata.org/prop/>
            PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
            PREFIX ps: <http://www.wikidata.org/prop/statement/>
            PREFIX wd: <http://www.wikidata.org/entity/>
            PREFIX wdt: <http://www.wikidata.org/prop/direct/>

            SELECT ?statement {{
              ?statement ps:P2302 wd:Q21510864. ## value-requires-statement constraint
              <{row['property']}> p:P2302 ?statement.
              ?statement pq:P2306/wikibase:directClaim <{row['wdt_required_property']}>.
              # exception and deprecation
              FILTER NOT EXISTS {{?statement pq:P2241 []}}
              FILTER NOT EXISTS {{?statement wikibase:rank wikibase:DeprecatedRank}}
              {req_val}
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        #print(response.text)

        # Parse the XML data
        namespace = {'sparql': 'http://www.w3.org/2005/sparql-results#'}
        root = ET.fromstring(response.text)

        # Find the uri inside the binding
        if root.find('.//sparql:binding[@name="statement"]/sparql:uri', namespace) is not None:
            return root.find('.//sparql:binding[@name="statement"]/sparql:uri', namespace).text
        return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
    
def getRequiredProperty(constraint_statement):
        
    #wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"
    
    # SPARQL query
    query = f"""
            PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
            PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX wikibase: <http://wikiba.se/ontology#>
            PREFIX p: <http://www.wikidata.org/prop/>
            PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
            PREFIX ps: <http://www.wikidata.org/prop/statement/>
            PREFIX wd: <http://www.wikidata.org/entity/>
            PREFIX wdt: <http://www.wikidata.org/prop/direct/>

            SELECT ?wdt_required_property {{
              <{constraint_statement}> pq:P2306/wikibase:directClaim ?wdt_required_property.
              # exception and deprecation
              FILTER NOT EXISTS {{<{constraint_statement}> pq:P2241 []}}
              FILTER NOT EXISTS {{<{constraint_statement}> wikibase:rank wikibase:DeprecatedRank}}
            }}
            """
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        #print(response.text)

        # Parse the XML data
        namespace = {'sparql': 'http://www.w3.org/2005/sparql-results#'}
        root = ET.fromstring(response.text)

        # Find the uri inside the binding
        if root.find('.//sparql:binding[@name="wdt_required_property"]/sparql:uri', namespace) is not None:
            return root.find('.//sparql:binding[@name="wdt_required_property"]/sparql:uri', namespace).text
        return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
def hasRequiredPropertyChanged(row):
    constraint_id_2019 = getConstraintID(row)
    if constraint_id_2019 is None:
        return None
    req_prop_2023 = getRequiredProperty(constraint_id_2019)
    if req_prop_2023 is None:
        return False
    if req_prop_2023 != row['wdt_required_property']:
        return True
    return False
        
    
# Example usage 
#hasRequiredPropertyChanged(df_vrs_repairs_cleaned.iloc[61485])
hasRequiredPropertyChanged(df_vrs_repairs_cleaned.iloc[367])

In [ ]:
df_vrs_repairs_cleaned['t-box required property changed'] = None

In [ ]:
start_index = df_vrs_repairs_cleaned[df_vrs_repairs_cleaned['t-box required property changed'].isna()].index.min()

start_index

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs_cleaned.iterrows(), total=len(df_vrs_repairs_cleaned)):
    
    # Check for unprocessed rows
    if pd.isna(row['t-box required property changed']):
        result = hasRequiredPropertyChanged(row)
        df_vrs_repairs_cleaned.at[index, 't-box required property changed'] = result

    # Save a checkpoint every 10,000 rows
    if index % 500000 == 0:
        df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_vrs_repairs_cleaned['t-box required property changed'].value_counts()

In [ ]:
df_vrs_repairs_cleaned['t-box required property changed'].value_counts()

In [ ]:
df_vrs_repairs_cleaned['t-box required property changed'].value_counts()

In [ ]:
df_vrs_repairs_cleaned.iloc[367]

In [ ]:
df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['Constraint Deleted'] == False)
    & (df_vrs_repairs_cleaned['Constraint Deprecated'] == False)
    & (df_vrs_repairs_cleaned['Included as Exception'] == False)
    & (df_vrs_repairs_cleaned['A-box wdt statement Deleted'] == False)
    & (df_vrs_repairs_cleaned['t-box required property changed'] == False)
    & ((df_vrs_repairs_cleaned['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['A-box wdt required statement added'])))
]['2019_no_req_val'].value_counts()

In [ ]:
df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['Constraint Deleted'] == False)
    & (df_vrs_repairs_cleaned['Constraint Deprecated'] == False)
    & (df_vrs_repairs_cleaned['Included as Exception'] == False)
    & (df_vrs_repairs_cleaned['A-box wdt statement Deleted'] == False)
    & (df_vrs_repairs_cleaned['t-box required property changed'] == False)
    & ((df_vrs_repairs_cleaned['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['A-box wdt required statement added'])))
]

In [ ]:
getConstraintID(df_vrs_repairs_cleaned.iloc[473531])

In [ ]:
import requests
import xml.etree.ElementTree as ET

def reqValStmtAdded(row):
    
    if row['object'].startswith("_:"):
        return None
    
    if row['2019_no_req_val'] is True:
        return None
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    
        ASK {{ 
          <{row['property']}> p:P2302 ?statement.
          ?statement ps:P2302 wd:Q21510864.
          ?statement pq:P2305 ?expected_val.
          <{row['object']}> <{row['wdt_required_property']}> ?expected_val.
        }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
print(reqValStmtAdded(df_vrs_repairs_cleaned.iloc[473531]))

In [ ]:
df_vrs_repairs_cleaned.to_csv("checkpoint_vrs_ok.csv", index=False)

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs_cleaned.iterrows(), total=len(df_vrs_repairs_cleaned)):
    
    # Check for unprocessed rows
    if bool(row['2019_no_req_val']) is False:
        result = reqValStmtAdded(row)
        df_vrs_repairs_cleaned.at[index, 'A-box wdt required statement added'] = result

    # Save a checkpoint every 10,000 rows
    if index % 100000 == 0:
        df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_vrs_repairs_cleaned[(df_vrs_repairs_cleaned['A-box wdt required statement added'] == True)] )

In [ ]:
df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['Constraint Deleted'] == False)
    & (df_vrs_repairs_cleaned['Constraint Deprecated'] == False)
    & (df_vrs_repairs_cleaned['Included as Exception'] == False)
    & (df_vrs_repairs_cleaned['A-box wdt statement Deleted'] == False)
    & (df_vrs_repairs_cleaned['t-box required property changed'] == False)
    & ((df_vrs_repairs_cleaned['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['A-box wdt required statement added'])))
]

In [ ]:
df_vrs_repairs_cleaned['t-box required value changed'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getCountRequiredValues(row, endpoint):
        
    #wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    # endpoint = "ENTER_qEndpoint_WD_2019"

    if bool(row['2019_no_req_val']) and  endpoint == "ENTER_qEndpoint_WD_2019":
        return 0
    
    
    # SPARQL query
    query = f"""
            PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
            PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX wikibase: <http://wikiba.se/ontology#>
            PREFIX p: <http://www.wikidata.org/prop/>
            PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
            PREFIX ps: <http://www.wikidata.org/prop/statement/>
            PREFIX wd: <http://www.wikidata.org/entity/>
            PREFIX wdt: <http://www.wikidata.org/prop/direct/>

            SELECT (COUNT(?expected_values) as ?total) {{
              ?statement ps:P2302 wd:Q21510864. ## value-requires-statement constraint
              <{row['property']}> p:P2302 ?statement.
              ?statement pq:P2306/wikibase:directClaim <{row['wdt_required_property']}>.
              # exception and deprecation
              FILTER NOT EXISTS {{?statement pq:P2241 []}}
              FILTER NOT EXISTS {{?statement wikibase:rank wikibase:DeprecatedRank}}
              ?statement pq:P2305 ?expected_values.
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:

        # Parse the XML data
        namespace = {'sparql': 'http://www.w3.org/2005/sparql-results#'}
        root = ET.fromstring(response.text)

        # Find the uri inside the binding
        if root.find('.//sparql:binding[@name="total"]/sparql:literal', namespace) is not None:
            return root.find('.//sparql:binding[@name="total"]/sparql:literal', namespace).text
        return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
def hasRequiredValuesNumberIncreased(row):
    n_2019 = getCountRequiredValues(row,"ENTER_qEndpoint_WD_2019")
    if n_2019 is None:
        return None
    n_2023 = getCountRequiredValues(row,"ENTER_qEndpoint_WD_2023")
    if n_2023 is None:
        return None
    if int(n_2023) > int(n_2019):
        return True
    return False
    
#print(getCountRequiredValues(df_vrs_repairs_cleaned.iloc[497101],"ENTER_qEndpoint_WD_2019"))
#print(getCountRequiredValues(df_vrs_repairs_cleaned.iloc[497101],"ENTER_qEndpoint_WD_2023"))
#hasRequiredValuesNumberIncreased(df_vrs_repairs_cleaned.iloc[497101])

In [ ]:
df_vrs_repairs_cleaned['t-box required value changed'] =  None

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs_cleaned.iterrows(), total=len(df_vrs_repairs_cleaned)):
    
    if pd.isna(row['t-box required value changed']) or row['t-box required value changed'] is None:
        if bool(row['Constraint Deleted']) is True:
            df_vrs_repairs_cleaned.at[index, 't-box required value changed'] = False
        else:
            result = hasRequiredValuesNumberIncreased(row)
            df_vrs_repairs_cleaned.at[index, 't-box required value changed'] = result

    # Save a checkpoint every 10,000 rows
    if index % 100000 == 0 and index != 0:
        df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_vrs_repairs_cleaned[(df_vrs_repairs_cleaned['t-box required value changed'] == True)] )

In [ ]:
df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['Constraint Deleted'] == False)
    & (df_vrs_repairs_cleaned['Constraint Deprecated'] == False)
    & (df_vrs_repairs_cleaned['Included as Exception'] == False)
    & (df_vrs_repairs_cleaned['A-box wdt statement Deleted'] == False)
    & (df_vrs_repairs_cleaned['t-box required value changed'] == False)
    & (df_vrs_repairs_cleaned['t-box required property changed'] == False)
    & ((df_vrs_repairs_cleaned['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['A-box wdt required statement added'])))
]

In [ ]:
getConstraintID(df_vrs_repairs_cleaned.iloc[1296137])

In [ ]:
getConstraintID(df_vrs_repairs_cleaned.iloc[1296137], "ENTER_qEndpoint_WD_2023")

In [ ]:
import requests
import xml.etree.ElementTree as ET

def constraintDeprecated(constraint_id):
    
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{constraint_id}> ?p ?o 
            FILTER (?p = wikibase:rank)
            FILTER (?o = wikibase:DeprecatedRank)
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    #print(query)
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

def constraintDeleted(constraint_id):
    
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{constraint_id}> ?p ?o 
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    #print(query)
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
    
# Example usage
print(constraintDeprecated(getConstraintID(df_vrs_repairs_cleaned.iloc[1296137])))
print(constraintDeleted(getConstraintID(df_vrs_repairs_cleaned.iloc[1296137])))

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs_cleaned.iterrows(), total=len(df_vrs_repairs_cleaned)):
    
    constraint_id = getConstraintID(row)
    if bool(row['Constraint Deprecated']) is False:
        result = constraintDeprecated(constraint_id)
        df_vrs_repairs_cleaned.at[index, 'Constraint Deprecated'] = result
        
    if bool(row['Constraint Deleted']) is False:
            result = constraintDeleted(constraint_id)
            df_vrs_repairs_cleaned.at[index, 'Constraint Deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 300000 == 0 and index != 0:
        df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:

import pandas as pd

# Read the first CSV file into a DataFrame
df_vrs_repairs = pd.read_csv("checkpoint_vrs.csv")

df_vrs_repairs

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['Constraint Deprecated'] == True)] )


In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['Constraint Deleted'] == True)] )

In [ ]:
df_vrs_repairs[
    (df_vrs_repairs['Constraint Deleted'] == False)
    & (df_vrs_repairs['Constraint Deprecated'] == False)
    & (df_vrs_repairs['Included as Exception'] == False)
    & (df_vrs_repairs['A-box wdt statement Deleted'] == False)
    & (df_vrs_repairs['t-box required value changed'] == False)
    & (df_vrs_repairs['t-box required property changed'] == False)
    & ((df_vrs_repairs['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs['A-box wdt required statement added'])))
]

In [ ]:
df_vrs_repairs.to_csv("vrs_repairs_all.csv", index=False)

In [ ]:
df_vrs_repairs.dtypes

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['A-box wdt required statement added'] == True)] )

In [ ]:
len(df_vrs_repairs)

In [ ]:
df_vrs_repairs

In [ ]:
df_vrs_repairs['t-box required value removed'] = None

In [ ]:
def requiredValueRemoved(row):
    n_2019 = getCountRequiredValues(row,"ENTER_qEndpoint_WD_2019")
    if n_2019 is None:
        return None
    n_2023 = getCountRequiredValues(row,"ENTER_qEndpoint_WD_2023")
    if n_2023 is None:
        return None
    if int(n_2023) == 0 and int(n_2019) > 0:
        return True
    return False

In [ ]:
requiredValueRemoved(df_vrs_repairs.iloc[1345664])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    if pd.isna(row['t-box required value removed']) or row['t-box required value removed'] is None:
        if bool(row['Constraint Deleted']) is True:
            df_vrs_repairs.at[index, 't-box required value removed'] = False
        else:
            result = requiredValueRemoved(row)
            df_vrs_repairs.at[index, 't-box required value removed'] = result

    # Save a checkpoint every 10,000 rows
    if index % 100000 == 0 and index != 0:
        df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_vrs_repairs['t-box required value removed'].value_counts()

In [ ]:
df_vrs_repairs[
    (df_vrs_repairs['Constraint Deleted'] == False)
    & (df_vrs_repairs['Constraint Deprecated'] == False)
    & (df_vrs_repairs['Included as Exception'] == False)
    & (df_vrs_repairs['A-box wdt statement Deleted'] == False)
    & (df_vrs_repairs['t-box required value changed'] == False)
    & (df_vrs_repairs['t-box required property changed'] == False)
    & ((df_vrs_repairs['A-box wdt required statement added'] == False) 
       | (pd.isna(df_vrs_repairs['A-box wdt required statement added'])))
]

In [ ]:
df_vrs_repairs.to_csv("vrs_all_repairs.csv", index=False)